In [1]:
from data import AttentionPolicy
from state import State, vectorize, tokenize
from data import load_games, prepare_sequence_data

path = "./data/DATABASE4U.pgn"
mode = "sequence" # or cnn
save_path = f"./data/processed/database4u_withturn_{mode}.npz"
vectors, actions, results = load_games(path, mode, max_length=160, save_path=save_path)

--- Loaded cache from ./data/processed/database4u_withturn_sequence.npz ---


In [2]:
vectors, results = vectors[:100], results[:100]

print(vectors[1].tolist())
State.deserialize(vectors[1])

[4286, 4275, 4273, 4274, 4276, 4277, 4274, 4273, 4275, 4288, 4272, 4272, 4272, 4272, 4272, 4272, 4272, 4272, 4288, 4289, 4289, 4289, 4289, 4289, 4289, 4289, 4289, 4288, 4289, 4289, 4289, 4289, 4289, 4289, 4289, 4289, 4288, 4289, 4289, 4289, 4289, 4278, 4289, 4289, 4289, 4288, 4289, 4289, 4289, 4289, 4289, 4289, 4289, 4289, 4288, 4278, 4278, 4278, 4278, 4289, 4278, 4278, 4278, 4288, 4281, 4279, 4280, 4282, 4283, 4280, 4279, 4281, 4287, 4290, 4015, 4013, 3690, 3688, 3567, 3502, 3437, 3372, 3307, 3242, 3177, 3112, 3559, 3494, 3429, 3364, 3299, 3234, 3169, 3104, 4291, 4292, 4293, 4294, 4285, 796, 4284, 4297, 3307, 4295, 4295, 4295, 4295, 4295, 4295, 4295, 4295, 4295, 4295, 4295, 4295, 4295, 4295, 4295, 4295, 4295, 4295, 4295, 4295, 4295, 4295, 4295, 4295, 4295, 4295, 4295, 4295, 4295, 4295, 4295, 4295, 4295, 4295, 4295, 4295, 4295, 4295, 4295, 4295, 4295, 4295, 4295, 4295, 4295, 4295, 4295, 4295, 4295, 4295, 4295, 4295, 4295, 4295, 4295, 4295, 4295]


'rnbqkbnr/pppppppp/8/8/4P3/8/PPPP1PPP/RNBQKBNR<legal_moves>g8h6g8f6b8c6b8a6h7h6g7g6f7f6e7e6d7d6c7c6b7b6a7a6h7h5g7g5f7f5e7e5d7d5c7c5b7b5a7a5<white_king><white_queen><black_king><black_queen><opponent>e2e4<me><black_turn>d7d6<pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad>'

In [3]:
import torch

model = AttentionPolicy(32, 2, True)
w = torch.load("./model/tiny_10.pth", weights_only=True)
w = {k: v for k, v in w.items() if "mask" not in k}
model.load_state_dict(w, strict=False)

_IncompatibleKeys(missing_keys=['blocks.0.mask', 'blocks.1.mask'], unexpected_keys=[])

In [8]:
after_e2e4 = "rnbqkbnr/pppppppp/8/8/4P3/8/PPPP1PPP/RNBQKBNR"
state = State.from_fen(after_e2e4)
print(state)
x = prepare_sequence_data(state.serialize(), vectorize("e2e4")) + [4297]
# x = prepare_sequence_data(state.serialize(), vectorize("e2e4"))
print(State.deserialize(x))
x = torch.tensor(x).unsqueeze(0)
tok = model(x).squeeze(0).argmax(dim=1).tolist()
print(x)
print([tokenize(token) for token in tok])
print(tok)


r n b q k b n r
p p p p p p p p
. . . . . . . .
. . . . . . . .
. . . . P . . .
. . . . . . . .
P P P P . P P P
R N B Q K B N R
rnbqkbnr/pppppppp/8/8/4P3/8/PPPP1PPP/RNBQKBNR<legal_moves>g1h3g1f3g1e2f1a6f1b5f1c4f1d3f1e2e1e2d1h5d1g4d1f3d1e2b1c3b1a3e4e5h2h3g2g3f2f3d2d3c2c3b2b3a2a3h2h4g2g4f2f4d2d4c2c4b2b4a2a4<opponent>e2e4<me><black_turn>
['c3e4', 'e8e7', 'd8b6', 'f3g5', 'f8d6', 'b1c3', 'f3g5', 'd8b6', 'e8e7', 'c1f4', 'e7f5', 'e7f5', 'e7f5', 'e7f5', 'e7f5', 'e7f5', 'e7f5', 'e7f5', 'c1f4', 'e2d3', 'e2d3', 'e2d3', 'e2d3', 'e2d3', 'e2d3', 'e2d3', 'e2d3', 'c1f4', 'e2d3', 'e2d3', 'e2d3', 'e2d3', 'e2d3', 'e2d3', 'e2d3', 'e2d3', 'c1f4', 'e2d3', 'e2d3', 'e2d3', 'e2d3', 'e1f1', 'e2d3', 'e2d3', 'e2d3', 'c1f4', 'e2d3', 'e2d3', 'e2d3', 'e2d3', 'e2d3', 'e2d3', 'e2d3', 'e2d3', 'c1f4', 'e1f1', 'e1f1', 'e1f1', 'e1f1', 'e2d3', 'e1f1', 'e1f1', 'e1f1', 'c1f4', 'g4e2', 'd8e8', 'f1a6', 'e8f8', 'h7h5', 'f1a6', 'd8e8', 'g4e2', 'd3e2', 'g1h3', 'g1f3', 'g1e2', 'f1a6', 'f1b5', 'f1c4', 'f1d3', 'f1e2', 'e1e2', 'd1h5'

In [17]:
from state import tokenize
tokenize(4292)

'<white_queen>'